In [1]:
import MDAnalysis as mda
from MDAnalysis.analysis import distances, contacts, rms

import numpy as np
import pandas as pd

In [2]:
# Fonction qui trouve le nombre d'atomes dans le peptide associé au WD40
def find_atoms_numbers(gro_file):
    
    with open(gro_file, "r") as f:
        lines = f.readlines()
    
    atom_lines = lines[2:-1]
    atoms = 0
    
    for line in atom_lines:
        atoms += 1
        if line.startswith("  166ARG      N"): # NOTE: ne fonctionne que pour le domaine WD40 murin de TBL1R!
            return(int(atoms-1))
            

In [3]:
# Fonction qui calcule le nombre de contacts entre 2 groupes et dans le rayon définis pour chaque frame de la trajectoire
def contacts_within_cutoff(u, group_a, group_b, radius=4.5):
    
    timeseries = []
    
    for ts in u.trajectory:
        # Calcule les distances entre les groupes A et B
        dist = contacts.distance_array(group_a.positions, group_b.positions)
        # Détermine le nombres de contacts (<= radius)
        n_contacts = contacts.contact_matrix(dist, radius).sum()
        timeseries.append([ts.frame, ts.frame/100, n_contacts])
        
    return np.array(timeseries)
    

In [4]:
def sel_atomes(u, protein_begin):
        
    protein_end = protein_begin + 5251 # NOTE: ne fonctionne que pour le domaine WD40 murin de TBL1R!

    # Définis la partie MRD et WD40 
    MRD = u.atoms[0:protein_begin]
    WD40 = u.atoms[protein_begin:protein_end]

    # Sélection des atomes polaires
    sel_basic = "(resname ARG and name NE NH* HE HH1* HH2*) or (resname LYS and name NZ HZ*) or (resname HIS and name ND1 NE2 HD1 HE2)"
    sel_acidic = "name OE* OD* O"
    basic = MRD.select_atoms(sel_basic)
    acidic = WD40.select_atoms(sel_acidic)

    # Sélection des résidus du MRD:
    sel_H325 = "(resid 325 and name ND1 NE2 HE2)"
    sel_R328 = "(resid 328 and name NE NH* HE HH1* HH2*)"
    sel_K326 = "(resid 326 and name NZ HZ*)"
    sel_K330 = "(resid 330 and name NZ HZ*)"
    sel_K332 = "(resid 332 and name NZ HZ*)"
    
    H325 = MRD.select_atoms(sel_H325)
    R328 = MRD.select_atoms(sel_R328)
    K326 = MRD.select_atoms(sel_K326)
    K330 = MRD.select_atoms(sel_K330)
    K332 = MRD.select_atoms(sel_K332)

    return acidic, basic, H325, R328, K326, K330, K332
    

In [5]:
# Fonction qui merged tous les contacts de chaque résidus dans une seule DF
def search_contacts(acidic, basic, H325, R328, K326, K330, K332, u_gro, radius=4.5):
    
    ca_all = contacts_within_cutoff(u_gro, acidic, basic, radius=radius)
    ca_df_all = pd.DataFrame(ca_all, columns=["Frame", "Time (ns)", "All_Contacts"])
    
    ca_H325 = contacts_within_cutoff(u_gro, acidic, H325, radius=radius)
    ca_df_H325 = pd.DataFrame(ca_H325, columns=["Frame", "Time (ns)", "H325_Contacts"])
    merged_df = pd.merge(ca_df_all, ca_df_H325, on=["Frame", "Time (ns)"])
    
    ca_R328 = contacts_within_cutoff(u_gro, acidic, R328, radius=radius)
    ca_df_R328 = pd.DataFrame(ca_R328, columns=["Frame", "Time (ns)", "R328_Contacts"])
    merged_df = pd.merge(merged_df, ca_df_R328, on=["Frame", "Time (ns)"])
    
    ca_K326 = contacts_within_cutoff(u_gro, acidic, K326, radius=radius)
    ca_df_K326 = pd.DataFrame(ca_K326, columns=["Frame", "Time (ns)", "K326_Contacts"])
    merged_df = pd.merge(merged_df, ca_df_K326, on=["Frame", "Time (ns)"])
    
    ca_K330 = contacts_within_cutoff(u_gro, acidic, K330, radius=radius)
    ca_df_K330 = pd.DataFrame(ca_K330, columns=["Frame", "Time (ns)", "K330_Contacts"])
    merged_df = pd.merge(merged_df, ca_df_K330, on=["Frame", "Time (ns)"])
    
    ca_K332 = contacts_within_cutoff(u_gro, acidic, K332, radius=radius)
    ca_df_K332 = pd.DataFrame(ca_K332, columns=["Frame", "Time (ns)", "K332_Contacts"])
    merged_df = pd.merge(merged_df, ca_df_K332, on=["Frame", "Time (ns)"])
    
    return merged_df
    

In [6]:
# Fonction qui sélectionne les résidus et compte chaques intéraction sur toutes les différentes versions des trajectoires WT
def trajectory(xtc_path, gro_path, versions, mutation = False, name = [""]):
    if mutation == False:
        for version in versions:
            # Lance la trajectoire de MD
            GRO = gro_path+version+"/md_0_1.gro"
            XTC = xtc_path+version+"/md_0_1_noPBC.xtc"
            u_gro = mda.Universe(GRO, XTC)
    
            # Définition des atomes du MRD
            peptide_count = find_atoms_numbers(GRO)
            print(peptide_count)
    
            # Sélection des atomes 
            acidic, basic, H325, R328, K326, K330, K332 = sel_atomes(u_gro, peptide_count)
    
            # Recherche des contacts et sauvegarde
            merged = search_contacts(acidic, basic, H325, R328, K326, K330, K332, u_gro)
            merged.to_csv(f"{gro_path}{version}/merged_contacts_{version}.csv", sep=";")
            print(f"DF sauvegardé dans: {gro_path}{version}/merged_contacts_{version}.csv")
    
    else:
        for version in versions:
            for mut in name:
                # Lance la trajectoire de MD
                GRO = gro_path+version+"/"+mut+"/md_0_1.gro"
                XTC = xtc_path+version+"/"+mut+"/md_0_1_noPBC.xtc"
                u_gro = mda.Universe(GRO, XTC)

                # Définitions des atomes du MRD
                peptide_count = find_atoms_numbers(GRO)
                print(peptide_count)

                # Sélection des atomes
                acidic, basic, H325, R328, K326, K330, K332 = sel_atomes(u_gro, peptide_count)

                # Recherche des contacts et sauvegarde
                merged = search_contacts(acidic, basic, H325,  R328, K326, K330, K332, u_gro)  
                merged.to_csv(f"{gro_path}{version}/{mut}/merged_contacts_{mut}_{version}.csv", sep=";")
                print(f"DF sauvegardé dans: {gro_path}{version}/{mut}/merged_contacts_{mut}_{version}.csv")


In [7]:
# Fonction qui moyenne les DF merged
def average_df(path, versions, mutation=False, name=[""]):

    if mutation == False:
        merged_df1 = pd.read_csv(f"{path}V1/merged_contacts_V1.csv", sep=";", index_col=0)
        merged_df2 = pd.read_csv(f"{path}V2/merged_contacts_V2.csv", sep=";", index_col=0)
        merged_df3 = pd.read_csv(f"{path}V3/merged_contacts_V3.csv", sep=";", index_col=0)

        avg_df = pd.concat([merged_df1, merged_df2, merged_df3]).groupby(["Frame", "Time (ns)"], as_index=True).mean()
        avg_df.to_csv(f"{path}averaged_contacts.csv", sep=";")
        print(f"DF moyenné dans: {path}averaged_contacts.csv")        
    
    else:
        for mut in name:
            merged_df1 = pd.read_csv(f"{path}V1/{mut}/merged_contacts_{mut}_V1.csv", sep=";", index_col=0).reset_index()
            merged_df2 = pd.read_csv(f"{path}V2/{mut}/merged_contacts_{mut}_V2.csv", sep=";", index_col=0).reset_index()
            merged_df3 = pd.read_csv(f"{path}V3/{mut}/merged_contacts_{mut}_V3.csv", sep=";", index_col=0).reset_index()
            
            avg_df = pd.concat([merged_df1, merged_df2, merged_df3]).groupby(["Frame", "Time (ns)"], as_index=True).mean()
            avg_df.to_csv(f"{path}averaged_contacts_{mut}.csv", sep=";")
            print(f"DF moyenné dans: {path}averaged_contacts_{mut}.csv")


In [8]:
# Variables:

In [9]:
input_path_wt = "E:/Trajectoire_MD/mNFIL3_100ns/WT/"
output_path_wt = "C:/Users/e216138h/Desktop/Stage/Dossier_Florent/M2/Modelisation/Partenaire/TBL1R/DM/new_MD/mNFIL3_100ns/WT/"

input_path_mut = "E:/Trajectoire_MD/mNFIL3_100ns/Mutations/"
output_path_mut = "C:/Users/e216138h/Desktop/Stage/Dossier_Florent/M2/Modelisation/Partenaire/TBL1R/DM/new_MD/mNFIL3_100ns/Mutations/"

input_path_wt_dist = "E:/Trajectoire_MD/mNFIL3_100ns/Distance/WT/"
output_path_wt_dist = "C:/Users/e216138h/Desktop/Stage/Dossier_Florent/M2/Modelisation/Partenaire/TBL1R/DM/new_MD/mNFIL3_100ns/Distance/WT/"

input_path_mut_dist = "E:/Trajectoire_MD/mNFIL3_100ns/Distance/Mutations/"
ouput_path_mut_dist = "C:/Users/e216138h/Desktop/Stage/Dossier_Florent/M2/Modelisation/Partenaire/TBL1R/DM/new_MD/mNFIL3_100ns/Distance/Mutations/"

In [10]:
replicas = ["V1", "V2", "V3"]
name_mutations = ["H325A", "I329A", "K326A", "K330A", "K332A", "L323A", "L327A", "P324A", "R328A", "S321A"]

In [11]:
# Lancement des fonctions

In [12]:
# trajectory(input_path_wt, output_path_wt, replicas)
# trajectory(input_path_mut, output_path_mut, replicas, True, name_mutations)

In [13]:
# trajectory(input_path_wt_dist, output_path_wt_dist, replicas)
# trajectory(input_path_mut_dist, ouput_path_mut_dist, replicas, mutation=True, name=name_mutations)

In [14]:
# average_df(output_path_wt, replicas)
# average_df(output_path_mut, replicas, True, name_mutations)

In [15]:
# average_df(output_path_wt_dist, replicas)
# average_df(ouput_path_mut_dist, replicas, mutation=True, name=name_mutations)